# Chapter 19 - Ablation: can the GNN carry the graph on its own?

Hands-On Graph Neural Networks Using Python, Second Edition.

In the main experiment (`train_colab.ipynb` and `run.py`), the LLM receives each explanation graph twice: through the GNN soft prompt and as text in the prompt. This notebook trains two more models, with the same data, seed, and hyperparameters, in which the graph is *not* given as text:

| Model | GNN soft prompt | Graph as text in the prompt |
|---|---|---|
| G-Retriever, GNN only | yes | no |
| LoRA, no graph | no | no |

It requires a CUDA GPU runtime. Run the cells in order; the last cell downloads the trained weights. `ablation.py` evaluates both models on the test set and compares them with the results of `run.py`.

## 1. Install the dependencies and check the GPU

In [1]:
!pip install -q 'torch_geometric==2.8.0' transformers accelerate peft sentencepiece
!pip install -q -U torchao
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.3 MB/s eta 0:00:00
Tue Oct  6 13:44:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 

## 2. Setup and dataset loader

In [2]:
import csv, json, math, os, random, re, time, urllib.request
import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.llm.models import LLM, GRetriever, SentenceTransformer
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GAT

SEED = 0
device = torch.device('cuda')
print(f'Device: {torch.cuda.get_device_name(0)}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_llm():
    llm = LLM(
        model_name='Qwen/Qwen3-0.6B',
        num_params=0.6,
        dtype=torch.bfloat16,
        sys_prompt=(
            '/no_think You are a helpful assistant. Given a belief, an '
            'argument, and a commonsense graph, decide whether the argument '
            'supports or counters the belief. Answer with a single word: '
            'support or counter.'
        ),
    )
    llm.llm.generation_config.do_sample = False
    llm.llm.generation_config.temperature = None
    llm.llm.generation_config.top_p = None
    llm.llm.generation_config.top_k = None
    return llm


DATA_DIR = './data/ExplaGraphs'
BASE = 'https://raw.githubusercontent.com/swarnaHub/ExplaGraphs/master/data'
os.makedirs(DATA_DIR, exist_ok=True)


def download_files():
    for name in ['train.tsv', 'dev.tsv']:
        local = os.path.join(DATA_DIR, name)
        if not os.path.exists(local):
            urllib.request.urlretrieve(f'{BASE}/{name}', local)


def parse_graph(graph_str):
    triples = []
    for match in re.findall(r'\(([^)]+)\)', graph_str):
        parts = [p.strip() for p in match.split(';')]
        if len(parts) == 3:
            triples.append(tuple(parts))
    return triples


def read_tsv(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        for r in reader:
            if len(r) < 4:
                continue
            belief, argument, stance, graph_str = r[:4]
            if stance not in ('support', 'counter'):
                continue
            triples = parse_graph(graph_str)
            if not triples:
                continue
            rows.append({'belief': belief.strip(),
                         'argument': argument.strip(),
                         'stance': stance.strip(),
                         'triples': triples})
    return rows


def build_dataset(rows, sbert):
    dataset = []
    for row in rows:
        nodes, node_ids, es, ed, er = [], {}, [], [], []
        for src, rel, dst in row['triples']:
            for txt in (src, dst):
                if txt not in node_ids:
                    node_ids[txt] = len(nodes)
                    nodes.append(txt)
            es.append(node_ids[src])
            ed.append(node_ids[dst])
            er.append(rel)

        x = sbert.encode(nodes, batch_size=64, output_device='cpu')
        edge_attr = sbert.encode(er, batch_size=64, output_device='cpu')

        s = torch.tensor(es, dtype=torch.long)
        d = torch.tensor(ed, dtype=torch.long)
        edge_index = torch.stack([torch.cat([s, d]), torch.cat([d, s])], dim=0)
        edge_attr = torch.cat([edge_attr, edge_attr], dim=0)

        desc = ' '.join(f'({s}; {r}; {d})' for s, r, d in row['triples'])
        question = (f"/no_think Belief: {row['belief']}\n"
                    f"Argument: {row['argument']}\n"
                    f"Does the argument support or counter the belief? "
                    f"Answer with a single word: support or counter.")

        dataset.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                            question=question, label=row['stance'], desc=desc))
    return dataset

Device: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 3. Build the dataset

In [3]:
set_seed(SEED)
download_files()
sbert = SentenceTransformer(
    'sentence-transformers/all-roberta-large-v1'
).to(device)
sbert.eval()

train_rows = read_tsv(os.path.join(DATA_DIR, 'train.tsv'))
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(train_rows))
split = int(0.8 * len(train_rows))
train_ds = build_dataset([train_rows[i] for i in idx[:split]], sbert)
val_ds = build_dataset([train_rows[i] for i in idx[split:]], sbert)

print(f'train={len(train_ds)}, val={len(val_ds)}')
support = sum(d.label == 'support' for d in val_ds)
print(f'val majority class (support): {support/len(val_ds):.3f}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

train=1894, val=474
val majority class (support): 0.584


## 4. Training function

In [4]:
LR, WD, MIN_LR, GRAD_CLIP = 1e-5, 0.05, 5e-6, 0.1
NUM_EPOCHS, WARMUP_EPOCHS = 10, 1
os.makedirs('./checkpoints', exist_ok=True)


def adjust_lr(pg, epoch, total):
    if epoch < WARMUP_EPOCHS:
        lr = LR
    else:
        prog = (epoch - WARMUP_EPOCHS) / max(1, total - WARMUP_EPOCHS)
        lr = MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * prog))
    pg['lr'] = lr
    return lr


def first_word(pred):
    answer = re.sub(r'<think>.*?</think>', '', pred, flags=re.DOTALL)
    words = answer.strip().lower().split()
    return re.sub(r'[^a-z]', '', words[0]) if words else ''


def is_correct(pred, label):
    return first_word(pred).startswith(label[:4].lower())


def compute_loss(model, batch, use_desc):
    batch = batch.to(device)
    return model(
        question=batch.question,
        x=batch.x.float(),
        edge_index=batch.edge_index,
        batch=batch.batch,
        label=batch.label,
        edge_attr=batch.edge_attr.float(),
        additional_text_context=batch.desc if use_desc else None,
    )


@torch.no_grad()
def evaluate(model, loader, use_desc):
    model.eval()
    total_loss, n_seen, correct = 0.0, 0, 0
    for batch in loader:
        loss = compute_loss(model, batch, use_desc)
        total_loss += loss.item() * batch.num_graphs
        n_seen += batch.num_graphs
        preds = model.inference(
            question=batch.question,
            x=batch.x.float().to(device),
            edge_index=batch.edge_index.to(device),
            batch=batch.batch.to(device),
            edge_attr=batch.edge_attr.float().to(device),
            additional_text_context=batch.desc if use_desc else None,
            max_out_tokens=24,
        )
        correct += sum(is_correct(p, l) for p, l in zip(preds, batch.label))
    model.train()
    return total_loss / n_seen, correct / n_seen


def train(model, name, use_desc=True):
    set_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WD,
    )
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    for epoch in range(NUM_EPOCHS):
        lr = adjust_lr(optimizer.param_groups[0], epoch, NUM_EPOCHS)
        model.train()
        train_loss, n_seen = 0.0, 0
        for batch in train_loader:
            optimizer.zero_grad()
            loss = compute_loss(model, batch, use_desc)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item() * batch.num_graphs
            n_seen += batch.num_graphs
        train_loss /= n_seen
        val_loss, val_acc = evaluate(model, val_loader, use_desc)
        print(f'--- Epoch {epoch+1}/{NUM_EPOCHS} ---')
        print(f'  lr={lr:.2e}  train_loss={train_loss:.4f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

    minutes = (time.time() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f'Training time: {minutes:.1f} min | Peak GPU memory: {peak:.1f} GB')

    state = {k: v.detach().cpu()
             for k, v in model.named_parameters() if v.requires_grad}
    path = f'./checkpoints/{name}_explagraphs.pt'
    torch.save({'epoch': NUM_EPOCHS, 'state_dict': state, 'val_acc': val_acc},
               path)
    print(f'Saved {path} ({os.path.getsize(path)/1e6:.1f} MB)')

## 5. Train G-Retriever without the graph as text (GNN only)

In [5]:
set_seed(SEED)
gnn = GAT(
    in_channels=1024,
    hidden_channels=1024,
    num_layers=4,
    out_channels=1024,
    heads=4,
    edge_dim=1024,
)
gnn_only = GRetriever(
    llm=make_llm(),
    gnn=gnn,
    use_lora=True,
    mlp_out_tokens=1,
).to(device)

train(gnn_only, 'gretriever_nodesc', use_desc=False)

Setting up 'Qwen/Qwen3-0.6B' with configuration: {'revision': 'main', 'max_memory': {0: '92GiB'}, 'low_cpu_mem_usage': True, 'device_map': 'auto', 'dtype': torch.bfloat16}


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

--- Epoch 1/10 ---
  lr=1.00e-05  train_loss=1.3960  val_loss=0.3836  val_acc=0.584
--- Epoch 2/10 ---
  lr=1.00e-05  train_loss=0.3131  val_loss=0.3038  val_acc=0.736
--- Epoch 3/10 ---
  lr=9.85e-06  train_loss=0.3174  val_loss=0.3895  val_acc=0.785
--- Epoch 4/10 ---
  lr=9.42e-06  train_loss=0.3325  val_loss=0.5809  val_acc=0.772
--- Epoch 5/10 ---
  lr=8.75e-06  train_loss=0.3587  val_loss=0.7213  val_acc=0.793
--- Epoch 6/10 ---
  lr=7.93e-06  train_loss=0.3506  val_loss=0.6111  val_acc=0.789
--- Epoch 7/10 ---
  lr=7.07e-06  train_loss=0.3270  val_loss=0.7213  val_acc=0.776
--- Epoch 8/10 ---
  lr=6.25e-06  train_loss=0.3007  val_loss=0.7937  val_acc=0.787
--- Epoch 9/10 ---
  lr=5.58e-06  train_loss=0.2907  val_loss=0.7485  val_acc=0.778
--- Epoch 10/10 ---
  lr=5.15e-06  train_loss=0.2597  val_loss=0.6982  val_acc=0.797
Training time: 5.6 min | Peak GPU memory: 7.4 GB
Saved ./checkpoints/gretriever_nodesc_explagraphs.pt (71.9 MB)


## 6. Train the LoRA model without any graph

In [6]:
set_seed(SEED)
no_graph = GRetriever(
    llm=make_llm(),
    gnn=None,
    use_lora=True,
).to(device)

train(no_graph, 'lora_nograph', use_desc=False)

Setting up 'Qwen/Qwen3-0.6B' with configuration: {'revision': 'main', 'max_memory': {0: '90GiB'}, 'low_cpu_mem_usage': True, 'device_map': 'auto', 'dtype': torch.bfloat16}


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

--- Epoch 1/10 ---
  lr=1.00e-05  train_loss=2.3743  val_loss=0.3324  val_acc=0.593
--- Epoch 2/10 ---
  lr=1.00e-05  train_loss=0.3037  val_loss=0.2844  val_acc=0.726
--- Epoch 3/10 ---
  lr=9.85e-06  train_loss=0.2689  val_loss=0.3585  val_acc=0.778
--- Epoch 4/10 ---
  lr=9.42e-06  train_loss=0.2975  val_loss=0.4696  val_acc=0.774
--- Epoch 5/10 ---
  lr=8.75e-06  train_loss=0.3409  val_loss=0.5377  val_acc=0.795
--- Epoch 6/10 ---
  lr=7.93e-06  train_loss=0.3473  val_loss=0.5827  val_acc=0.795
--- Epoch 7/10 ---
  lr=7.07e-06  train_loss=0.3339  val_loss=0.6368  val_acc=0.791
--- Epoch 8/10 ---
  lr=6.25e-06  train_loss=0.3135  val_loss=0.6618  val_acc=0.795
--- Epoch 9/10 ---
  lr=5.58e-06  train_loss=0.3011  val_loss=0.6942  val_acc=0.795
--- Epoch 10/10 ---
  lr=5.15e-06  train_loss=0.2868  val_loss=0.6608  val_acc=0.800
Training time: 5.3 min | Peak GPU memory: 9.7 GB
Saved ./checkpoints/lora_nograph_explagraphs.pt (4.6 MB)


## 7. Download the trained weights

In [7]:
from google.colab import files

for name in ['gretriever_nodesc', 'lora_nograph']:
    files.download(f'./checkpoints/{name}_explagraphs.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>